# Combined MS1 — data cleaning and auditing

Preserves shared work and the four member sections. Missing-value filling remains deferred. Flagged anomalies remain unresolved unless explicitly documented. Original CSVs and input notebooks are unchanged. See `acl-ms1-merge-notes.md` for integration changes.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Kaggle first; local archive is supported for verification.
DATA_DIR = Path("/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020")
if not DATA_DIR.exists():
    DATA_DIR = Path("archive")
if not DATA_DIR.exists():
    raise FileNotFoundError("Set DATA_DIR to the folder containing the F1 CSV files.")


In [ ]:
circuits =  pd.read_csv(DATA_DIR / "circuits.csv") 
constructor_results =  pd.read_csv(DATA_DIR / "constructor_results.csv") 
constructor_standings =  pd.read_csv(DATA_DIR / "constructor_standings.csv") 
constructors =  pd.read_csv(DATA_DIR / "constructors.csv") 
driver_standings =  pd.read_csv(DATA_DIR / "driver_standings.csv") 
drivers =  pd.read_csv(DATA_DIR / "drivers.csv") 
lap_times =  pd.read_csv(DATA_DIR / "lap_times.csv") 
pit_stops =  pd.read_csv(DATA_DIR / "pit_stops.csv") 
qualifying =  pd.read_csv(DATA_DIR / "qualifying.csv") 
races =  pd.read_csv(DATA_DIR / "races.csv") 
seasons =  pd.read_csv(DATA_DIR / "seasons.csv") 
sprint_results =  pd.read_csv(DATA_DIR / "sprint_results.csv") 
status =  pd.read_csv(DATA_DIR / "status.csv") 
results =  pd.read_csv(DATA_DIR / "results.csv") 

## Shared section — circuits and constructor results


## circuits.csv

One row identifies one circuit. `circuitId` is its primary key. There are no date, duration, or foreign-key columns.


In [ ]:

circuits.info()

In [ ]:
# Count recognized missing values and literal placeholders separately.
circuits_before_summary = pd.DataFrame({
    "Data type": circuits.dtypes.astype(str),
    "Recognized missing": circuits.isna().sum(),
    "Placeholders": circuits.eq(r"\N").sum()
})
display(circuits_before_summary)
circuits_before_summary[["Recognized missing", "Placeholders"]].plot.bar(figsize=(9, 4))
plt.title("circuits: unavailable values before cleaning")
plt.ylabel("Number of rows")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


The original `info()` output checks recognized missing values only. Literal `\N` placeholders require a separate check; no placeholders were found in the inspected circuits copy.


In [ ]:
# Display every row and column.
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(circuits)

In [ ]:

counts = circuits["country"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 10))

counts.plot.barh(ax=ax, color="teal")

ax.set_title("Country Labels Before Cleaning")
ax.set_xlabel("Number of circuits")
ax.set_ylabel("Country label")

plt.tight_layout()
plt.show()

In [ ]:
circuits_clean = circuits.copy(deep=True).replace(r"\N", np.nan)
for column in circuits_clean.select_dtypes(include="object").columns:
    circuits_clean[column] = circuits_clean[column].str.strip()
circuits_clean["country"] = circuits_clean["country"].replace({"USA": "United States"})
for column in ["lat", "lng", "alt"]:
    circuits_clean[column] = pd.to_numeric(circuits_clean[column], errors="raise")


In [ ]:
circuits_comparison = pd.DataFrame({
    "Placeholders BEFORE": circuits.eq(r"\N").sum(),
    "Placeholders AFTER": circuits_clean.eq(r"\N").sum(),
    "Recognized missing BEFORE": circuits.isna().sum(),
    "Recognized missing AFTER": circuits_clean.isna().sum()
})
display(circuits_comparison)
circuits_comparison[["Placeholders BEFORE", "Placeholders AFTER"]].plot.bar(figsize=(9, 4))
plt.title("circuits: placeholder handling before and after")
plt.ylabel("Number of placeholders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


Recognized missing counts can rise when placeholders are converted. This does not mean new information was lost. Zero placeholder counts before and after indicate that no replacement was needed in those columns.


In [ ]:
changed_countries = circuits["country"].fillna("<MISSING>").ne(circuits_clean["country"].fillna("<MISSING>"))
display(pd.DataFrame({
    "country BEFORE": circuits.loc[changed_countries, "country"],
    "country AFTER": circuits_clean.loc[changed_countries, "country"]
}))
print("Country labels changed:", changed_countries.sum())


In [ ]:

counts = circuits_clean["country"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 10))

counts.plot.barh(ax=ax, color="teal")

ax.set_title("Country Labels After Cleaning")
ax.set_xlabel("Number of circuits")
ax.set_ylabel("Country label")

plt.tight_layout()
plt.show()

In [ ]:
print("Missing circuit IDs:",
      circuits_clean["circuitId"].isna().sum())

print("Duplicate circuit IDs:",
      circuits_clean["circuitId"].duplicated().sum())

assert circuits_clean["circuitId"].notna().all(), "Missing circuit IDs found"
assert circuits_clean["circuitId"].is_unique, "Duplicate circuit IDs found"

print("Passed: circuitId is unique and non-null.")

### Circuits: final checks and findings

Country aliases are standardized so the same country is not counted twice. No missing values are filled. Negative coordinates or altitude are not automatically errors. Coordinate-range checks below test valid geographic limits, not precise location accuracy.


In [ ]:
circuits_checks = pd.Series({
    "Rows unchanged": len(circuits_clean) == len(circuits),
    "Columns unchanged": circuits_clean.columns.equals(circuits.columns),
    "No placeholders remain": not circuits_clean.eq(r"\N").any().any(),
    "No duplicate rows": not circuits_clean.duplicated().any(),
    "circuitId non-null": circuits_clean["circuitId"].notna().all(),
    "circuitId unique": circuits_clean["circuitId"].is_unique,
    "Latitude within bounds": circuits_clean["lat"].dropna().between(-90, 90).all(),
    "Longitude within bounds": circuits_clean["lng"].dropna().between(-180, 180).all()
})
display(circuits_checks.to_frame("Passed"))
assert circuits_checks.all(), "Investigate failed circuits checks."
circuits_clean.info()


## constructor_results.csv

One row records a constructor result for a race. The primary key is `constructorResultsId`; `(raceId, constructorId)` should also be unique. No date or duration conversion applies. Current-race points are not pre-race model inputs.


In [ ]:
constructor_results.head()

In [ ]:
# Count recognized missing values and literal placeholders separately.
constructor_results_before_summary = pd.DataFrame({
    "Data type": constructor_results.dtypes.astype(str),
    "Recognized missing": constructor_results.isna().sum(),
    "Placeholders": constructor_results.eq(r"\N").sum()
})
display(constructor_results_before_summary)
constructor_results_before_summary[["Recognized missing", "Placeholders"]].plot.bar(figsize=(9, 4))
plt.title("constructor_results: unavailable values before cleaning")
plt.ylabel("Number of rows")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
constructor_results_clean = constructor_results.copy(deep=True)
constructor_results_clean = constructor_results_clean.replace(r"\N", np.nan)
constructor_results_clean["points"] = pd.to_numeric(constructor_results_clean["points"], errors="raise")
display(constructor_results_clean.head())


In [ ]:
constructor_results_clean.isnull().sum()

In [ ]:
constructor_results_comparison = pd.DataFrame({
    "Placeholders BEFORE": constructor_results.eq(r"\N").sum(),
    "Placeholders AFTER": constructor_results_clean.eq(r"\N").sum(),
    "Recognized missing BEFORE": constructor_results.isna().sum(),
    "Recognized missing AFTER": constructor_results_clean.isna().sum()
})
display(constructor_results_comparison)
constructor_results_comparison[["Placeholders BEFORE", "Placeholders AFTER"]].plot.bar(figsize=(9, 4))
plt.title("constructor_results: placeholder handling before and after")
plt.ylabel("Number of placeholders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


Recognized missing counts can rise when placeholders are converted. This does not mean new information was lost. Zero placeholder counts before and after indicate that no replacement was needed in those columns.


In [ ]:


display(
    constructor_results_clean["status"]
    .value_counts(dropna=False)
    .rename_axis("Status")
    .reset_index(name="Number of rows")
)

# Display the special flagged records.
display(
    constructor_results_clean.loc[
        constructor_results_clean["status"].notna()
    ]
)

In [ ]:
# Check the record ID.
record_id = "constructorResultsId"

print("Missing constructorResultsId:",
      constructor_results_clean[record_id].isna().sum())

print("Duplicate constructorResultsId:",
      constructor_results_clean[record_id].duplicated().sum())

assert constructor_results_clean[record_id].notna().all(), \
    "Missing constructorResultsId found"

assert constructor_results_clean[record_id].is_unique, \
    "Duplicate constructorResultsId found"


# Check the race–constructor combination.
pair_columns = ["raceId", "constructorId"]

print("\nMissing race and constructor IDs:")
display(constructor_results_clean[pair_columns].isna().sum())

print("Duplicate race–constructor pairs:",
      constructor_results_clean.duplicated(pair_columns).sum())

assert constructor_results_clean[pair_columns].notna().all().all(), \
    "Missing raceId or constructorId found"

assert not constructor_results_clean.duplicated(pair_columns).any(), \
    "Duplicate race–constructor pairs found"

print("\nPassed: record IDs and race–constructor pairs are unique and non-null.")

In [ ]:
# Find rows with missing or unknown race IDs.
invalid_races = constructor_results_clean[
    constructor_results_clean["raceId"].isna()
    | ~constructor_results_clean["raceId"].isin(races["raceId"])
]

# Find rows with missing or unknown constructor IDs.
invalid_constructors = constructor_results_clean[
    constructor_results_clean["constructorId"].isna()
    | ~constructor_results_clean["constructorId"].isin(
        constructors["constructorId"]
    )
]

print("Rows with missing or unknown race IDs:", len(invalid_races))
print("Rows with missing or unknown constructor IDs:", len(invalid_constructors))

if not invalid_races.empty:
    display(invalid_races)

if not invalid_constructors.empty:
    display(invalid_constructors)

assert invalid_races.empty, \
    "Some race IDs are missing or absent from races"

assert invalid_constructors.empty, \
    "Some constructor IDs are missing or absent from constructors"

print("Passed: all race and constructor IDs exist in their reference tables.")

### Constructor results: final checks and findings

Literal `\N` entries are converted to recognized missing values without filling them. `status` is a special constructor-result flag, not the driver retirement reason from `status.csv`. Missing status entries and the `D` flag are preserved; their meaning must be verified before analytical use. The key and reference checks above validate identifiers, not the interpretation of that flag.


In [ ]:
constructor_results_checks = pd.Series({
    "Rows unchanged": len(constructor_results_clean) == len(constructor_results),
    "Columns unchanged": constructor_results_clean.columns.equals(constructor_results.columns),
    "No placeholders remain": not constructor_results_clean.eq(r"\N").any().any(),
    "No duplicate rows": not constructor_results_clean.duplicated().any(),
    "Points are numeric": pd.api.types.is_numeric_dtype(constructor_results_clean["points"]),
    "Points non-negative": constructor_results_clean["points"].dropna().ge(0).all()
})
display(constructor_results_checks.to_frame("Passed"))
assert constructor_results_checks.all(), "Investigate failed constructor-result checks."
constructor_results_clean.info()


# Rezk — Data cleaning and auditing

Scope: drivers, lap times, and pit stops. Original tables remain unchanged; cleaning uses copies. Missing-value filling is deferred. Suspected recording issues are flagged rather than corrected without verification.


# 1. drivers.csv

### Before-cleaning inspection

Each row represents one driver. `driverId` is the primary key.


In [ ]:
print("Rows:", len(drivers))
print("Columns:", len(drivers.columns))
display(drivers.head(10))
drivers.info()


### Placeholder handling and before/after evidence


In [ ]:
sentinels_before = drivers.eq(r"\N").sum()

display(
    sentinels_before.to_frame("Missing placeholders BEFORE")
)

In [ ]:

drivers_clean = drivers.copy(deep=True)
drivers_clean = drivers_clean.replace(r"\N", np.nan)

In [ ]:
comparison = pd.DataFrame({
    "Placeholders BEFORE": drivers.eq(r"\N").sum(),
    "Placeholders AFTER": drivers_clean.eq(r"\N").sum(),
    "Recognized missing BEFORE": drivers.isna().sum(),
    "Recognized missing AFTER": drivers_clean.isna().sum()
})

display(comparison)

In [ ]:
comparison[[
    "Placeholders BEFORE",
    "Placeholders AFTER"
]].plot.bar(figsize=(10, 4))

plt.title("Drivers: Missing Placeholders Before and After Cleaning")
plt.ylabel("Number of placeholders")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

The original data uses `\N` to represent missing information.
We counted these placeholders before cleaning, then replaced them with
recognized missing values in a copy. The comparison confirms that no `\N`
placeholders remain. Missing information was preserved, not guessed or filled.

### Nationality inspection and correction


In [ ]:
# Show every nationality label and its count.
nationality_counts = (
    drivers_clean["nationality"]
    .value_counts(dropna=False)
    .rename_axis("Nationality")
    .reset_index(name="Number of drivers")
)

# Quotes make surrounding spaces visible.
nationality_counts["Nationality"] = (
    nationality_counts["Nationality"].map(repr)
)

with pd.option_context("display.max_rows", None):
    display(nationality_counts)

In [ ]:
nationality_before = drivers["nationality"].copy()

# Remove surrounding spaces.
drivers_clean["nationality"] = (
    drivers_clean["nationality"].str.strip()
)

# Use one label for the same nationality.
drivers_clean["nationality"] = (
    drivers_clean["nationality"].replace({
        "Argentinian": "Argentine"
    })
)

In [ ]:
changed = (
    nationality_before.fillna("<MISSING>")
    != drivers_clean["nationality"].fillna("<MISSING>")
)

changes = drivers_clean.loc[
    changed, ["driverId", "forename", "surname"]
].copy()

# Quotes make the original trailing spaces visible.
changes["Nationality BEFORE"] = nationality_before.loc[changed].map(repr)
changes["Nationality AFTER"] = (
    drivers_clean.loc[changed, "nationality"].map(repr)
)

display(changes)
print("Rows changed:", changed.sum())

In [ ]:
labels = ["Argentine", "Argentinian", "Argentinian "]

nationality_comparison = pd.DataFrame({
    "Before": nationality_before.value_counts().reindex(
        labels, fill_value=0
    ),
    "After": drivers_clean["nationality"].value_counts().reindex(
        labels, fill_value=0
    )
})

nationality_comparison.index = [repr(label) for label in labels]

display(nationality_comparison)

nationality_comparison.plot.bar(
    figsize=(9, 4),
    color=["steelblue", "teal"]
)

plt.title("Nationality Labels Before and After Cleaning")
plt.xlabel("Nationality label")
plt.ylabel("Number of drivers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

We removed surrounding spaces from nationality labels and standardized
“Argentinian” to “Argentine” because they describe the same nationality.
The before/after table and chart show the affected records and label counts.
No driver records were removed.

### Birth-date typing and before/after evidence


In [ ]:
# Preserve values before this change.
dob_before = drivers["dob"].replace(r"\N", np.nan).copy()

converted_dob = pd.to_datetime(
    dob_before,
    format="%Y-%m-%d",
    errors="coerce"
)

# Find non-missing strings that could not be converted.
invalid_dates = dob_before.notna() & converted_dob.isna()

print("Invalid non-missing birth dates:", invalid_dates.sum())

if invalid_dates.any():
    display(
        drivers_clean.loc[
            invalid_dates,
            ["driverId", "forename", "surname", "dob"]
        ]
    )

assert not invalid_dates.any(), \
    "Investigate invalid birth dates before continuing."

drivers_clean["dob"] = converted_dob

In [ ]:
display(pd.DataFrame({
    "BEFORE": dob_before.head(),
    "AFTER": drivers_clean["dob"].head()
}))

print("Type BEFORE:", dob_before.dtype)
print("Type AFTER:", drivers_clean["dob"].dtype)

print("Missing dates BEFORE:", dob_before.isna().sum())
print("Missing dates AFTER:", drivers_clean["dob"].isna().sum())

Dates of birth were stored as text. We checked for invalid date strings
and converted valid values to datetime format. We compared missing counts
before and after to verify that conversion did not introduce new gaps.
This enables calculating each driver's age at a race once race dates are joined.

### Key and reference checks


In [ ]:
print("Missing driver IDs:",
      drivers_clean["driverId"].isna().sum())

print("Duplicate driver IDs:",
      drivers_clean["driverId"].duplicated().sum())

assert drivers_clean["driverId"].notna().all(), \
    "Missing driverId found"

assert drivers_clean["driverId"].is_unique, \
    "Duplicate driverId found"

print("Passed: driverId is unique and non-null.")

In [ ]:
reference_tables = {
    "results": results,
    "qualifying": qualifying,
    "driver_standings": driver_standings
}

checks = []

for name, table in reference_tables.items():
    missing = (
        table["driverId"].isna()
        | table["driverId"].eq(r"\N")
    )

    unknown = (
        ~missing
        & ~table["driverId"].isin(drivers_clean["driverId"])
    )

    checks.append({
        "Table": name,
        "Missing driver IDs": int(missing.sum()),
        "Unknown driver IDs": int(unknown.sum())
    })

    if (missing | unknown).any():
        print(f"Problem records in {name}:")
        display(table.loc[missing | unknown].head(20))

audit = pd.DataFrame(checks)
display(audit)

assert audit[["Missing driver IDs", "Unknown driver IDs"]].eq(0).all().all(), \
    "Some driver references need investigation"

print("Passed: all checked driver references exist in drivers_clean.")

In [ ]:
final_checks = pd.Series({
    "Row count unchanged": len(drivers_clean) == len(drivers),
    "No remaining placeholders": not drivers_clean.eq(r"\N").any().any(),
    "Driver IDs non-null": drivers_clean["driverId"].notna().all(),
    "Driver IDs unique": drivers_clean["driverId"].is_unique,
    "No completely duplicated rows": not drivers_clean.duplicated().any(),
    "Birth dates have datetime type":
        pd.api.types.is_datetime64_any_dtype(drivers_clean["dob"]),
    "Birth-date missing count unchanged":
        drivers_clean["dob"].isna().sum()
        == drivers["dob"].replace(r"\N", np.nan).isna().sum()
})

display(final_checks.to_frame("Passed"))
assert final_checks.all(), "Investigate failed checks."

### Drivers: findings and deferred work

`drivers` is the original table. `drivers_clean` contains recognized missing values, consistent nationality labels, and datetime values in `dob`. No columns were added or rows removed. Missing `number` and `code` values remain unfilled. Nationality-to-country mapping is a later analysis step. Driver–race uniqueness belongs to results and the final modeling table.


In [ ]:
drivers_clean.info()
assert drivers_clean.columns.equals(drivers.columns), "Unexpected added or removed driver columns"


# 2. lap_times.csv

### Before-cleaning inspection

Each row represents one recorded lap for a driver in a race. The key is `(raceId, driverId, lap)`; repeated driver–race pairs across laps are expected.


In [ ]:
print("Rows:", lap_times.shape[0])
print("Columns:", lap_times.shape[1])

display(lap_times.head(10))
lap_times.info()

In [ ]:
missing = lap_times.isna().sum()
sentinels = lap_times.eq(r"\N").sum()

lap_inspection = pd.DataFrame({
    "Data type": lap_times.dtypes.astype(str),
    "Recognized missing": missing,
    r"\N placeholders": sentinels,
    "Total unavailable": missing + sentinels,
    "Unavailable %": (
        (missing + sentinels) / len(lap_times) * 100
    ).round(2)
})

display(lap_inspection)

In [ ]:
unavailable = missing + sentinels

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.bar(
    unavailable.index,
    unavailable.values,
    color="steelblue"
)

ax.bar_label(bars, padding=3)
ax.set_title("Lap Times: Unavailable Values Before Cleaning")
ax.set_ylabel("Number of rows")
ax.set_ylim(0, max(1, unavailable.max()) * 1.2)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Convert `time` to numeric seconds

The CSV already supplies `milliseconds`. After checking its availability, divide by 1,000 and replace `time` in `lap_times_clean`. Original text stays in `lap_times`. An independent text conversion below verifies agreement; no columns are added.


In [ ]:
lap_times_clean = lap_times.copy(deep=True)
lap_times_clean = lap_times_clean.replace(r"\N", np.nan)


In [ ]:
# Verify milliseconds are available wherever the original time is present.
lap_times_clean["milliseconds"] = pd.to_numeric(lap_times_clean["milliseconds"], errors="raise")
assert not (lap_times_clean["time"].notna() & lap_times_clean["milliseconds"].isna()).any(), "A recorded time has no milliseconds."

# 1,000 milliseconds = 1 second. Replace time in the cleaned copy only.
lap_times_clean["time"] = lap_times_clean["milliseconds"] / 1000


In [ ]:
# Independently compare original text with the converted time.
lap_text = lap_times["time"].replace(r"\N", np.nan).astype("string")
lap_text = lap_text.where(lap_text.str.count(":").ne(1), "0:" + lap_text)
lap_text_seconds = pd.to_timedelta(lap_text, errors="raise").dt.total_seconds()
lap_matches = np.isclose(lap_text_seconds, lap_times_clean["time"], atol=0.001, rtol=0, equal_nan=True)
print("Original time values agreeing with converted seconds:", lap_matches.sum())
assert lap_matches.all(), "Original time and milliseconds disagree."


### Before/after values and types

Compare the same `time` column in the original and cleaned tables. Hour-based examples demonstrate parsing support, not that unusually long records have been verified.


In [ ]:
display(pd.DataFrame({
    "time BEFORE (text)": lap_times["time"].head(10),
    "time AFTER (seconds)": lap_times_clean["time"].head(10)
}))
print("time BEFORE type:", lap_times["time"].dtype)
print("time AFTER type:", lap_times_clean["time"].dtype)
hour_examples = lap_times["time"].astype("string").str.count(":").eq(2).fillna(False)
display(pd.DataFrame({
    "time BEFORE (text)": lap_times.loc[hour_examples, "time"],
    "time AFTER (seconds)": lap_times_clean.loc[hour_examples, "time"],
    "milliseconds": lap_times_clean.loc[hour_examples, "milliseconds"]
}).head(5))


The cleaned `time` column contains seconds calculated from `milliseconds / 1000`. We independently converted original text to seconds and checked agreement. This verifies conversion consistency, not underlying record accuracy. Original rows and columns are preserved; no missing values are filled.


### Key and reference checks


In [ ]:
key_columns = ["raceId", "driverId", "lap"]

print("Missing key values:")
display(lap_times_clean[key_columns].isna().sum())

print(
    "Duplicate race–driver–lap records:",
    lap_times_clean.duplicated(key_columns).sum()
)

# Display problem records, if any.
duplicates = lap_times_clean.duplicated(key_columns, keep=False)

if duplicates.any():
    display(
        lap_times_clean.loc[duplicates]
        .sort_values(key_columns)
        .head(20)
    )

assert lap_times_clean[key_columns].notna().all().all(), \
    "Missing raceId, driverId, or lap found"

assert not duplicates.any(), \
    "Repeated race–driver–lap records found"

print("Passed: race–driver–lap combinations are unique and non-null.")

In [ ]:
unknown_races = ~lap_times_clean["raceId"].isin(races["raceId"])
unknown_drivers = ~lap_times_clean["driverId"].isin(drivers["driverId"])

print("Rows with unknown race IDs:", unknown_races.sum())
print("Rows with unknown driver IDs:", unknown_drivers.sum())

if unknown_races.any():
    display(lap_times_clean.loc[unknown_races].head(20))

if unknown_drivers.any():
    display(lap_times_clean.loc[unknown_drivers].head(20))

assert not unknown_races.any(), \
    "Some race IDs do not exist in races"

assert not unknown_drivers.any(), \
    "Some driver IDs do not exist in drivers"

print("Passed: all race and driver IDs exist in their reference tables.")

In [ ]:
# Deduplicate only the lookup pairs, not the lap records.
result_pairs = results[["raceId", "driverId"]].drop_duplicates()

lap_pairs = (
    lap_times_clean[["raceId", "driverId"]]
    .drop_duplicates()
)

pair_check = lap_pairs.merge(
    result_pairs,
    on=["raceId", "driverId"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

unmatched_pairs = pair_check.loc[
    pair_check["_merge"].eq("left_only"),
    ["raceId", "driverId"]
]

print("Driver–race pairs absent from results:", len(unmatched_pairs))

if not unmatched_pairs.empty:
    display(unmatched_pairs.head(20))

assert unmatched_pairs.empty, \
    "Some lap records belong to driver–race pairs absent from results"

print("Passed: all lap driver–race pairs exist in results.")

The combination of raceId, driverId, and lap is unique and non-null.
All race and driver references exist, and every recorded driver–race pair
has a corresponding result record. Repeated driver–race pairs across
different laps are expected and were preserved.

### Unusual-value investigation

Long recorded durations are inspected, not automatically removed. Their cause is not established by their size alone.


In [ ]:
display(
    lap_times_clean[
        ["lap", "position", "time"]
    ].describe().T
)

print("Rows with lap <= 0:",
      lap_times_clean["lap"].le(0).sum())

print("Rows with position <= 0:",
      lap_times_clean["position"].le(0).sum())

print("Rows with time <= 0:",
      lap_times_clean["time"].le(0).sum())

In [ ]:
# Logarithmic bins show typical and unusually long durations together.
values = lap_times_clean.loc[lap_times_clean["time"] > 0, "time"].dropna()
if not values.empty:
    bins = np.geomspace(values.min(), values.max(), 40) if values.min() < values.max() else 10
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(values, bins=bins, color="teal", edgecolor="white")
    ax.set_xscale("log")
    ax.set_title("Lap times: time in seconds")
    ax.set_xlabel("time (seconds, logarithmic scale)")
    ax.set_ylabel("Number of records")
    plt.tight_layout()
    plt.show()


In [ ]:
print("Largest time values — flagged for investigation, not removed:")
display(
    lap_times_clean.nlargest(10, "time")[
        ["raceId", "driverId", "lap", "position",
         "time", "milliseconds"]
    ]
)

### Final verification


In [ ]:
final_checks = pd.Series({
    "Row count unchanged":
        len(lap_times_clean) == len(lap_times),

    "No remaining placeholders":
        not lap_times_clean.eq(r"\N").any().any(),

    "Key columns non-null":
        lap_times_clean[["raceId", "driverId", "lap"]]
        .notna().all().all(),

    "Race–driver–lap key unique":
        not lap_times_clean.duplicated(
            ["raceId", "driverId", "lap"]
        ).any(),

    "time is numeric seconds":
        pd.api.types.is_numeric_dtype(
            lap_times_clean["time"]
        )
})

display(final_checks.to_frame("Passed"))

assert final_checks.all(), "Investigate failed checks."

### Lap times: cleaning and audit

- Preserved the original table and cleaned a copy.
- Inspected missing values and placeholder values before cleaning.
- Converted recorded lap durations into numeric seconds.
- Verified conversion against the milliseconds column.
- Checked uniqueness and non-nullness of the race–driver–lap key.
- Verified race, driver, and driver–race references.
- Examined duration distributions and the longest records.
- Preserved unusually long durations pending investigation.
- Did not fill missing values or remove lap records.
- Current-race lap information will not be used as pre-race prediction inputs.

In [ ]:
lap_times_clean.info()
assert lap_times_clean.columns.equals(lap_times.columns), "Unexpected added or removed lap columns"


# 3. pit_stops.csv

### Before-cleaning inspection

Each row represents one recorded pit stop. The key is `(raceId, driverId, stop)`. `time` is a clock time; `duration` is elapsed time.


In [ ]:
print("Rows:", pit_stops.shape[0])
print("Columns:", pit_stops.shape[1])

display(pit_stops.head(10))
pit_stops.info()

In [ ]:
pit_missing = pit_stops.isna().sum()
pit_sentinels = pit_stops.eq(r"\N").sum()

pit_inspection = pd.DataFrame({
    "Data type": pit_stops.dtypes.astype(str),
    "Recognized missing": pit_missing,
    r"\N placeholders": pit_sentinels,
    "Total unavailable": pit_missing + pit_sentinels,
    "Unavailable %": (
        (pit_missing + pit_sentinels) / len(pit_stops) * 100
    ).round(2)
})

display(pit_inspection)

In [ ]:
unavailable = pit_missing + pit_sentinels

fig, ax = plt.subplots(figsize=(9, 4))

bars = ax.bar(
    unavailable.index,
    unavailable.values,
    color="steelblue"
)

ax.bar_label(bars, padding=3)
ax.set_title("Pit Stops: Unavailable Values Before Cleaning")
ax.set_ylabel("Number of rows")
ax.set_ylim(0, max(1, unavailable.max()) * 1.2)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
display(
    pit_stops.select_dtypes(include="number").describe().T
)

# Show examples of durations written with minutes.
minute_format = (
    pit_stops["duration"]
    .astype("string")
    .str.contains(":", na=False)
)

print("Durations containing a colon:", minute_format.sum())

display(
    pit_stops.loc[
        minute_format,
        ["raceId", "driverId", "stop", "lap",
         "duration", "milliseconds"]
    ].head(10)
)

### Placeholder handling and semantic typing


In [ ]:
pit_stops_clean = pit_stops.copy(deep=True)

pit_stops_clean = pit_stops_clean.replace(r"\N", np.nan)

In [ ]:
# Verify milliseconds are available wherever the original duration is present.
pit_stops_clean["milliseconds"] = pd.to_numeric(pit_stops_clean["milliseconds"], errors="raise")
assert not (pit_stops_clean["duration"].notna() & pit_stops_clean["milliseconds"].isna()).any(), "A recorded duration has no milliseconds."

pit_stops_clean["duration"] = pit_stops_clean["milliseconds"] / 1000


In [ ]:
# time is a clock reading, not an elapsed duration. Keep its text format.
# Validate the format without changing the column or inventing a date.
clock_check = pd.to_datetime(pit_stops_clean["time"], format="%H:%M:%S", errors="coerce")
invalid_clock = pit_stops_clean["time"].notna() & clock_check.isna()
print("Invalid non-missing time values:", invalid_clock.sum())
assert not invalid_clock.any(), "Investigate invalid clock readings."


In [ ]:
# Independently check both original duration formats against milliseconds.
pit_text = pit_stops["duration"].replace(r"\N", np.nan).astype("string")
pit_text = pit_text.where(pit_text.str.count(":").ne(1), "0:" + pit_text)
pit_text = pit_text.where(pit_text.str.contains(":", na=True), pit_text + "s")
pit_text_seconds = pd.to_timedelta(pit_text, errors="raise").dt.total_seconds()
pit_matches = np.isclose(pit_text_seconds, pit_stops_clean["duration"], atol=0.001, rtol=0, equal_nan=True)
print("Original duration values agreeing with converted seconds:", pit_matches.sum())
assert pit_matches.all(), "Original duration and milliseconds disagree."


### Before/after evidence


In [ ]:
display(pd.DataFrame({
    "duration BEFORE (text)": pit_stops["duration"].head(10),
    "duration AFTER (seconds)": pit_stops_clean["duration"].head(10),
    "time BEFORE (text)": pit_stops["time"].head(10),
    "time AFTER (clock text unchanged)": pit_stops_clean["time"].head(10)
}))
print("duration BEFORE type:", pit_stops["duration"].dtype)
print("duration AFTER type:", pit_stops_clean["duration"].dtype)
print("time BEFORE type:", pit_stops["time"].dtype)
print("time AFTER type:", pit_stops_clean["time"].dtype)
print("Rows BEFORE:", len(pit_stops))
print("Rows AFTER:", len(pit_stops_clean))
print("Missing duration AFTER:", pit_stops_clean["duration"].isna().sum())
print("Missing time AFTER:", pit_stops_clean["time"].isna().sum())


The cleaned `duration` contains numeric seconds from `milliseconds / 1000`, checked independently against the original text. `time` is a clock reading, not a duration: its text is preserved and its format validated. No date or timezone is inferred. Original tables remain unchanged; no columns are added and missing values are not filled.


### Key and reference checks


In [ ]:
key_columns = ["raceId", "driverId", "stop"]

display(
    pit_stops_clean[key_columns]
    .isna()
    .sum()
    .to_frame("Missing key values")
)

duplicates = pit_stops_clean.duplicated(
    key_columns, keep=False
)

print("Extra duplicate key rows:",
      pit_stops_clean.duplicated(key_columns).sum())

if duplicates.any():
    display(
        pit_stops_clean.loc[duplicates]
        .sort_values(key_columns)
        .head(20)
    )

assert pit_stops_clean[key_columns].notna().all().all(), \
    "Missing key values found"

assert not duplicates.any(), \
    "Repeated race–driver–stop combinations found"

print("Passed: the key is unique and non-null.")

In [ ]:
unknown_races = ~pit_stops_clean["raceId"].isin(races["raceId"])
unknown_drivers = ~pit_stops_clean["driverId"].isin(drivers["driverId"])

print("Rows with unknown race IDs:", unknown_races.sum())
print("Rows with unknown driver IDs:", unknown_drivers.sum())

if unknown_races.any():
    display(pit_stops_clean.loc[unknown_races].head(20))

if unknown_drivers.any():
    display(pit_stops_clean.loc[unknown_drivers].head(20))

assert not unknown_races.any(), "Unknown race IDs found"
assert not unknown_drivers.any(), "Unknown driver IDs found"

print("Passed: race and driver references exist.")

In [ ]:
pit_pairs = pit_stops_clean[
    ["raceId", "driverId"]
].drop_duplicates()

result_pairs = results[
    ["raceId", "driverId"]
].drop_duplicates()

pair_check = pit_pairs.merge(
    result_pairs,
    on=["raceId", "driverId"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

unmatched_pairs = pair_check.loc[
    pair_check["_merge"].eq("left_only"),
    ["raceId", "driverId"]
]

print("Driver–race pairs absent from results:", len(unmatched_pairs))

if not unmatched_pairs.empty:
    display(unmatched_pairs)

assert unmatched_pairs.empty, \
    "Some pit-stop driver–race pairs are absent from results"

print("Passed: all driver–race pairs exist in results.")

### Unusual stop-number investigation

The sequence check flags groups for review; it does not establish that a record is wrong or authorize renumbering.


In [ ]:
stop_summary = (
    pit_stops_clean.groupby(["raceId", "driverId"])
    .agg(
        recorded_stops=("stop", "size"),
        first_stop_number=("stop", "min"),
        last_stop_number=("stop", "max")
    )
    .reset_index()
)

# With unique positive integer stop numbers, these checks identify
# sequences that do not run continuously from 1 to the record count.
unusual_sequences = stop_summary[
    stop_summary["first_stop_number"].ne(1)
    | stop_summary["last_stop_number"].ne(
        stop_summary["recorded_stops"]
    )
]

print("Driver–race groups with unusual stop numbering:",
      len(unusual_sequences))

display(unusual_sequences.head(20))

print("Largest recorded stop numbers:")
display(
    pit_stops_clean.nlargest(10, "stop")[
        ["raceId", "driverId", "stop", "lap", "duration"]
    ]
)

In [ ]:
# Show all records in the flagged driver–race groups.
flagged_stops = pit_stops_clean.merge(
    unusual_sequences[["raceId", "driverId"]],
    on=["raceId", "driverId"],
    how="inner",
    validate="many_to_one"
)

display(
    flagged_stops[[
        "raceId", "driverId", "stop", "lap",
        "time", "duration"
    ]].sort_values(["raceId", "driverId", "time"])
)

### Duration investigation

A long recorded duration may reflect unusual race circumstances. Keep it pending verification.


In [ ]:
# Logarithmic bins show typical and unusually long durations together.
values = pit_stops_clean.loc[pit_stops_clean["duration"] > 0, "duration"].dropna()
if not values.empty:
    bins = np.geomspace(values.min(), values.max(), 40) if values.min() < values.max() else 10
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(values, bins=bins, color="teal", edgecolor="white")
    ax.set_xscale("log")
    ax.set_title("Pit stops: duration in seconds")
    ax.set_xlabel("duration (seconds, logarithmic scale)")
    ax.set_ylabel("Number of records")
    plt.tight_layout()
    plt.show()

print("Rows with duration <= 0:", pit_stops_clean["duration"].le(0).sum())
display(pit_stops_clean.nlargest(10, "duration")[["raceId", "driverId", "stop", "lap", "duration"]])


### Pit-stop audit findings and deferred work

Duration parsing, milliseconds consistency, key uniqueness, and reference checks are implemented above. The eight flagged driver–race groups in the inspected dataset have unusual stop numbering. Several race 1128 records suggest swapped `stop` and `lap` values; other groups begin at stop 2. These are suspected issues, not confirmed corrections. No records were renumbered, invented, or deleted. Long durations remain pending investigation. Missing-value filling is deferred. Current-race pit-stop information must not be used for pre-race prediction.


In [ ]:
pit_final_checks = pd.Series({
    "Row count unchanged": len(pit_stops_clean) == len(pit_stops),
    "No remaining placeholders": not pit_stops_clean.eq(r"\N").any().any(),
    "Key non-null": pit_stops_clean[["raceId", "driverId", "stop"]].notna().all().all(),
    "Key unique": not pit_stops_clean.duplicated(["raceId", "driverId", "stop"]).any(),
    "Clock text unchanged": pit_stops_clean["time"].equals(pit_stops["time"].replace(r"\N", np.nan)),
    "duration is numeric seconds": pd.api.types.is_numeric_dtype(pit_stops_clean["duration"]),
    "Duration missingness preserved": pit_stops_clean["duration"].isna().sum() == pit_stops["duration"].replace(r"\N", np.nan).isna().sum(),
    "Clock missingness preserved": pit_stops_clean["time"].isna().sum() == pit_stops["time"].replace(r"\N", np.nan).isna().sum()
})
display(pit_final_checks.to_frame("Passed"))
assert pit_final_checks.all(), "Investigate failed pit-stop checks."


In [ ]:
pit_stops_clean.info()
assert pit_stops_clean.columns.equals(pit_stops.columns), "Unexpected added or removed pit columns"


# Mohamed — constructor standings, constructors, driver standings


## Constructor standings


In [ ]:
constructor_standings.head()

In [ ]:
constructor_standings_clean  =  pd.read_csv(DATA_DIR / "constructor_standings.csv",na_values=[r"\N"]) 
constructor_standings_clean.head()

In [ ]:
constructor_standings_clean.info()

In [ ]:
# Check the record ID.
record_id = "constructorStandingsId"

print("Missing constructorStandingsId:",
      constructor_standings_clean[record_id].isna().sum())

print("Duplicate constructorStandingsId:",
      constructor_standings_clean[record_id].duplicated().sum())

assert constructor_standings_clean[record_id].notna().all(), \
    "Missing constructorStandingsId found"

assert constructor_standings_clean[record_id].is_unique, \
    "Duplicate constructorStandingsId found"


# Check the race–constructor combination.
pair_columns = ["raceId", "constructorId"]

print("\nMissing race and constructor IDs:")
display(constructor_standings_clean[pair_columns].isna().sum())

print("Duplicate race–constructor pairs:",
      constructor_standings_clean.duplicated(pair_columns).sum())

assert constructor_standings_clean[pair_columns].notna().all().all(), \
    "Missing raceId or constructorId found"

assert not constructor_standings_clean.duplicated(pair_columns).any(), \
    "Duplicate race–constructor pairs found"

print("\nPassed: record IDs and race–constructor pairs are unique and non-null.")

In [ ]:
# Find rows with missing or unknown race IDs.
invalid_races = constructor_standings_clean[
    constructor_standings_clean["raceId"].isna()
    | ~constructor_standings_clean["raceId"].isin(races["raceId"])
]

# Find rows with missing or unknown constructor IDs.
invalid_constructors = constructor_standings_clean[
    constructor_standings_clean["constructorId"].isna()
    | ~constructor_standings_clean["constructorId"].isin(
        constructors["constructorId"]
    )
]

print("Rows with missing or unknown race IDs:", len(invalid_races))
print("Rows with missing or unknown constructor IDs:", len(invalid_constructors))

if not invalid_races.empty:
    display(invalid_races)

if not invalid_constructors.empty:
    display(invalid_constructors)

assert invalid_races.empty, \
    "Some race IDs are missing or absent from races"

assert invalid_constructors.empty, \
    "Some constructor IDs are missing or absent from constructors"

print("Passed: all race and constructor IDs exist in their reference tables.")

## Constructors


In [ ]:
constructors.head()

In [ ]:
constructors_clean  =  pd.read_csv(DATA_DIR / "constructors.csv",na_values=[r"\N"]) 
constructors_clean.head()

In [ ]:
constructors_clean.info()

In [ ]:
# Check the record ID.
record_id = "constructorId"

print("Missing constructorId:",
      constructors_clean[record_id].isna().sum())

print("Duplicate constructorId:",
      constructors_clean[record_id].duplicated().sum())

assert constructors_clean[record_id].notna().all(), \
    "Missing constructorId found"

assert constructors_clean[record_id].is_unique, \
    "Duplicate constructorId found"


## Driver standings


In [ ]:
driver_standings.head()

In [ ]:
driver_standings_clean  =  pd.read_csv(DATA_DIR / "driver_standings.csv",na_values=[r"\N"]) 
driver_standings_clean.head()

In [ ]:
driver_standings_clean.info()

In [ ]:
# Check the record ID.
record_id = "driverStandingsId"

print("Missing driverStandingsId:",
      driver_standings_clean[record_id].isna().sum())

print("Duplicate driverStandingsId:",
      driver_standings_clean[record_id].duplicated().sum())

assert driver_standings_clean[record_id].notna().all(), \
    "Missing driverStandingsId found"

assert driver_standings_clean[record_id].is_unique, \
    "Duplicate driverStandingsId found"


# Check the race–constructor combination.
pair_columns = ["raceId", "driverId"]

print("\nMissing race and driver IDs:")
display(driver_standings_clean[pair_columns].isna().sum())

print("Duplicate race–driver pairs:",
      driver_standings_clean.duplicated(pair_columns).sum())

assert driver_standings_clean[pair_columns].notna().all().all(), \
    "Missing raceId or driverID found"

assert not driver_standings_clean.duplicated(pair_columns).any(), \
    "Duplicate race–driver pairs found"

print("\nPassed: record IDs and race–drivers pairs are unique and non-null.")

In [ ]:
# Find rows with missing or unknown race IDs.
invalid_races = driver_standings_clean[
    driver_standings_clean["raceId"].isna()
    | ~driver_standings_clean["raceId"].isin(races["raceId"])
]

# Find rows with missing or unknown constructor IDs.
invalid_drivers = driver_standings_clean[
    driver_standings_clean["driverId"].isna()
    | ~driver_standings_clean["driverId"].isin(
        drivers["driverId"]
    )
]

print("Rows with missing or unknown race IDs:", len(invalid_races))
print("Rows with missing or unknown drivers IDs:", len(invalid_drivers))

if not invalid_races.empty:
    display(invalid_races)

if not invalid_drivers.empty:
    display(invalid_drivers)

assert invalid_races.empty, \
    "Some race IDs are missing or absent from races"

assert invalid_drivers.empty, \
    "Some drivers IDs are missing or absent from drivers"

print("Passed: all race and drivers IDs exist in their reference tables.")

# Kareem — qualifying, races, results


### **qualifying table**

In [ ]:
print("shape:", qualifying.shape)
print("columns:", list(qualifying.columns))
qualifying.head()

In [ ]:
before = pd.DataFrame({
    "sentinel (\\N)": (qualifying == r"\N").sum(),
    "blank ('')":     (qualifying == "").sum(),
})
before["total missing"] = before.sum(axis=1)
before["% missing"] = (100 * before["total missing"] / len(qualifying)).round(2)
before

In [ ]:
##removed placeholders to null values

qualifying_clean =  pd.read_csv(DATA_DIR / "qualifying.csv", na_values=[r"\N"]) 
qualifying_clean.head()

In [ ]:
qualifying_clean.isnull().sum()

In [ ]:
check = pd.DataFrame({
    "missing_before": before["total missing"],
    "NaN_after": qualifying_clean.isnull().sum(),
})
check["match"] = check["missing_before"] == check["NaN_after"]
check

In [ ]:
def to_seconds(t):
    if pd.isna(t):
        return np.nan
    minutes, seconds = str(t).split(":")
    return int(minutes) * 60 + float(seconds)

for col in ["q1", "q2", "q3"]:
    qualifying_clean[col + "_sec"] = qualifying_clean[col].map(to_seconds)

qualifying_clean[["q1", "q1_sec", "q2", "q2_sec", "q3", "q3_sec"]].head()

In [ ]:
for col in ["q1", "q2", "q3"]:
    lost = (qualifying_clean[col].notna() & qualifying_clean[col + "_sec"].isna()).sum()
    print(col, "failed conversions:", lost)

In [ ]:
n_dup = qualifying_clean.duplicated(subset=["raceId", "driverId"]).sum()
print("duplicate (raceId, driverId) pairs:", n_dup)

In [ ]:
print("qualifyId unique  :", qualifying_clean["qualifyId"].is_unique)
print("qualifyId no nulls:", qualifying_clean["qualifyId"].notna().all())

In [ ]:
races_ids        = pd.read_csv(DATA_DIR / "races.csv", usecols=["raceId"])["raceId"]
drivers_ids      = pd.read_csv(DATA_DIR / "drivers.csv", usecols=["driverId"])["driverId"]
constructors_ids = pd.read_csv(DATA_DIR / "constructors.csv", usecols=["constructorId"])["constructorId"]

rows = []
for col, parent, ids in [("raceId", "races.raceId", races_ids),
                         ("driverId", "drivers.driverId", drivers_ids),
                         ("constructorId", "constructors.constructorId", constructors_ids)]:
    orphans = int((~qualifying_clean[col].dropna().isin(ids)).sum())
    rows.append({"foreign key": f"qualifying.{col}", "points to": parent,
                 "null values": int(qualifying_clean[col].isna().sum()),
                 "orphan rows": orphans, "ok": orphans == 0})
pd.DataFrame(rows)

### **races table**

In [ ]:
print("shape:", races.shape)
print("columns:", list(races.columns))
races.head()

In [ ]:
before_r = pd.DataFrame({"sentinel (\\N)": (races == r"\N").sum()})
before_r["% missing"] = (100 * before_r["sentinel (\\N)"] / len(races)).round(2)
before_r

In [ ]:
before_r["% missing"].plot(kind="bar", color="tab:red", figsize=(9, 3.5),
                           title="races (raw): % of values equal to \\N")
plt.ylabel("%")
plt.show()

In [ ]:
races_clean = pd.read_csv(DATA_DIR / "races.csv", keep_default_na=False, na_values=[r"\N"])
races_clean.head()

In [ ]:
check_r = pd.DataFrame({"sentinel_before": before_r["sentinel (\\N)"],
                        "NaN_after": races_clean.isnull().sum()})
check_r["match"] = check_r["sentinel_before"] == check_r["NaN_after"]
check_r

In [ ]:
date_cols = [c for c in ["date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"]
             if c in races_clean.columns]

for col in date_cols:
    n_before = races_clean[col].isna().sum()
    races_clean[col] = pd.to_datetime(races_clean[col], errors="coerce")
    print(col, "| new missing values created:", races_clean[col].isna().sum() - n_before)

races_clean[date_cols].dtypes

In [ ]:
n_before = races_clean["time"].isna().sum()
races_clean["race_start_time"] = pd.to_timedelta(races_clean["time"])
print("new missing values created:", races_clean["race_start_time"].isna().sum() - n_before)
races_clean[["time", "race_start_time"]].dropna().head()

In [ ]:
print("raceId unique  :", races_clean["raceId"].is_unique)
print("raceId no nulls:", races_clean["raceId"].notna().all())
print("(year, round) unique:", not races_clean.duplicated(["year", "round"]).any())

In [ ]:
circuits_ids = pd.read_csv(DATA_DIR / "circuits.csv", usecols=["circuitId"])["circuitId"]
seasons_ids  = pd.read_csv(DATA_DIR / "seasons.csv", usecols=["year"])["year"]

rows = []
for col, parent, ids in [("circuitId", "circuits.circuitId", circuits_ids),
                         ("year", "seasons.year", seasons_ids)]:
    orphans = int((~races_clean[col].dropna().isin(ids)).sum())
    rows.append({"foreign key": f"races.{col}", "points to": parent,
                 "null values": int(races_clean[col].isna().sum()),
                 "orphan rows": orphans, "ok": orphans == 0})
pd.DataFrame(rows)

In [ ]:
print("raceId order matches date order:", races_clean.sort_values("raceId")["date"].is_monotonic_increasing)

rounds = races_clean.groupby("year")["round"].agg(n_races="count", first="min", last="max")
rounds["gap"] = rounds["last"] - rounds["n_races"]
print("seasons with a gap or not starting at round 1:")
rounds[(rounds["first"] != 1) | (rounds["gap"] != 0)]

In [ ]:
races_clean.groupby("year").size().plot(kind="bar", figsize=(12, 3.5), title="Races per season")
plt.ylabel("races")
plt.show()

In [ ]:
import seaborn as sns

sched = [c for c in ["time", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date"]
         if c in races_clean.columns]
miss_dec = (races_clean.assign(decade=(races_clean["year"] // 10) * 10)
                       .groupby("decade")[sched]
                       .apply(lambda d: d.isna().mean() * 100))

sns.heatmap(miss_dec, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% missing"})
plt.title("races: % missing per column by decade")
plt.show()

In [ ]:
pd.DataFrame({
    "rows": [len(races), len(races_clean)],
    "columns": [races.shape[1], races_clean.shape[1]],
    "missing values": [int(before_r["sentinel (\\N)"].sum()), int(races_clean.isnull().sum().sum())],
    "text '\\N' left": [int(before_r["sentinel (\\N)"].sum()), 0],
}, index=["before", "after"])

### **Results table** 

In [ ]:
print("shape:", results.shape)
print("columns:", list(results.columns))
results.head()

In [ ]:
before_res = pd.DataFrame({"sentinel (\\N)": (results == r"\N").sum()})
before_res["% missing"] = (100 * before_res["sentinel (\\N)"] / len(results)).round(2)
before_res

In [ ]:
before_res["% missing"].plot(kind="bar", color="tab:red", figsize=(10, 3.5),
                             title="results (raw): % of values equal to \\N")
plt.ylabel("%")
plt.show()

In [ ]:
results_clean = pd.read_csv(DATA_DIR / "results.csv", keep_default_na=False, na_values=[r"\N"])
results_clean.head()

In [ ]:
check_res = pd.DataFrame({"sentinel_before": before_res["sentinel (\\N)"],
                          "NaN_after": results_clean.isnull().sum()})
check_res["match"] = check_res["sentinel_before"] == check_res["NaN_after"]
check_res

In [ ]:
results_clean["fastestLapTime_sec"] = results_clean["fastestLapTime"].map(to_seconds)

lost = (results_clean["fastestLapTime"].notna() & results_clean["fastestLapTime_sec"].isna()).sum()
print("failed conversions:", lost)
results_clean[["fastestLapTime", "fastestLapTime_sec"]].dropna().head()

In [ ]:
results_clean[["time", "milliseconds"]].dropna().head(10)

In [ ]:
print("positionOrder missing:", results_clean["positionOrder"].isna().sum(), "(must be 0)")
print("positionOrder < 1    :", (results_clean["positionOrder"] < 1).sum(), "(must be 0)")
print("grid range:", results_clean["grid"].min(), "->", results_clean["grid"].max())
print("negative points:", (results_clean["points"] < 0).sum())

In [ ]:
n_dup = results_clean.duplicated(subset=["raceId", "driverId"]).sum()
print("extra rows (same driver twice in the same race):", n_dup)

In [ ]:
dup_rows = results_clean[results_clean.duplicated(["raceId", "driverId"], keep=False)].sort_values(["raceId", "driverId", "resultId"])
print("rows involved:", len(dup_rows))
dup_rows[["resultId", "raceId", "driverId", "constructorId", "grid", "positionOrder", "points", "laps", "statusId"]].head(20)

In [ ]:
years = pd.read_csv(DATA_DIR / "races.csv", usecols=["raceId", "year"])
dup_plot = dup_rows.merge(years, on="raceId", how="left")     # copy for plotting only

dup_plot.groupby("year").size().plot(kind="bar", color="tab:orange", figsize=(10, 3.5),
                                     title="results (raw): rows involved in a duplicate, per season")
plt.ylabel("rows")
plt.show()

# In how many duplicated pairs did exactly one of the rows start from a real grid slot (grid > 0)?
started_per_pair = dup_rows.groupby(["raceId", "driverId"])["grid"].apply(lambda g: (g > 0).sum())
print("number of started rows per duplicated pair:")
started_per_pair.value_counts()

In [ ]:
results_clean["started"] = results_clean["grid"] > 0

results_clean = (results_clean
                 .sort_values(["raceId", "driverId", "started", "resultId"], ascending=[True, True, False, True])
                 .drop_duplicates(["raceId", "driverId"], keep="first")
                 .drop(columns="started")
                 .sort_values("resultId")
                 .reset_index(drop=True))

print("rows after:", len(results_clean), "| removed:", len(results) - len(results_clean))

In [ ]:
assert not results_clean.duplicated(["raceId", "driverId"]).any()
print("OK: (raceId, driverId) is unique")

In [ ]:
print("resultId unique  :", results_clean["resultId"].is_unique)
print("resultId no nulls:", results_clean["resultId"].notna().all())

In [ ]:
races_ids        = pd.read_csv(DATA_DIR / "races.csv", usecols=["raceId"])["raceId"]
drivers_ids      = pd.read_csv(DATA_DIR / "drivers.csv", usecols=["driverId"])["driverId"]
constructors_ids = pd.read_csv(DATA_DIR / "constructors.csv", usecols=["constructorId"])["constructorId"]
status_ids       = pd.read_csv(DATA_DIR / "status.csv", usecols=["statusId"])["statusId"]

rows = []
for col, parent, ids in [("raceId", "races.raceId", races_ids),
                         ("driverId", "drivers.driverId", drivers_ids),
                         ("constructorId", "constructors.constructorId", constructors_ids),
                         ("statusId", "status.statusId", status_ids)]:
    orphans = int((~results_clean[col].dropna().isin(ids)).sum())
    rows.append({"foreign key": f"results.{col}", "points to": parent,
                 "null values": int(results_clean[col].isna().sum()),
                 "orphan rows": orphans, "ok": orphans == 0})
pd.DataFrame(rows)

In [ ]:
import seaborn as sns

res_year = results_clean.merge(years, on="raceId", how="left")     # copy for plotting only
cols = ["number", "position", "milliseconds", "fastestLap", "rank", "fastestLapTime"]
miss_dec = (res_year.assign(decade=(res_year["year"] // 10) * 10)
                    .groupby("decade")[cols]
                    .apply(lambda d: d.isna().mean() * 100))

sns.heatmap(miss_dec, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "% missing"})
plt.title("results (clean): % missing per column by decade")
plt.show()

In [ ]:
before_rows = pd.read_csv(DATA_DIR / "results.csv", usecols=["raceId"]).merge(years, on="raceId").groupby("year").size()
after_rows  = res_year.groupby("year").size()

plt.figure(figsize=(11, 3.5))
plt.plot(before_rows.index, before_rows.values, label="raw", alpha=0.7)
plt.plot(after_rows.index, after_rows.values, label="clean")
plt.title("Driver-race rows per season")
plt.legend()
plt.show()

In [ ]:
pd.DataFrame({
    "rows": [len(results), len(results_clean)],
    "columns": [results.shape[1], results_clean.shape[1]],
    "missing values": [int(before_res["sentinel (\\N)"].sum()), int(results_clean.isnull().sum().sum())],
    "text '\\N' left": [int(before_res["sentinel (\\N)"].sum()), 0],
    "duplicate (raceId, driverId)": [n_dup, 0],
}, index=["before", "after"])

### Duplicate-result handling rule

The retained implementation prefers a row with `grid > 0`, then the smallest `resultId`. This is a deterministic selection rule, not a reliable definition of whether a driver started: zero grid can represent special starts or other historical cases. The flag is used only for duplicate selection and removed afterward. The original `results` table preserves all records; assess the effect on labels and historical shared drives before finalizing the modeling table.


# Hamza — seasons, sprint results, status


**seasons.csv**

In [ ]:
seasons_clean = pd.read_csv(DATA_DIR / "seasons.csv",na_values=[r"\N"]) 

In [ ]:
seasons_clean.head()

In [ ]:
seasons_clean.isnull().sum()

**sprint_results.csv**

In [ ]:
sprint_results_clean =  pd.read_csv(DATA_DIR / "sprint_results.csv",na_values=[r"\N"])


In [ ]:
sprint_results.isnull().sum()

In [ ]:
sprint_results_clean.isnull().sum()

In [ ]:
sprint_results_clean.info()

In [ ]:
sprint_results.head()

In [ ]:
sprint_results_clean.head()

In [ ]:
def time_to_seconds(x):
    if pd.isna(x):
        return np.nan
    parts = str(x).strip().lstrip("+").split(":")
    if not 1 <= len(parts) <= 3:
        raise ValueError(f"Unexpected duration: {x}")
    values = [float(part) for part in parts]
    if not all(np.isfinite(v) and v >= 0 for v in values) or any(v >= 60 for v in values[1:]):
        raise ValueError(f"Invalid duration: {x}")
    seconds = 0.0
    for value in values:
        seconds = seconds * 60 + value
    return seconds


In [ ]:
# Preserve original strings. Total times and gaps have different meanings.
sprint_time_text = sprint_results_clean["time"].astype("string").str.strip()
sprint_is_gap = sprint_time_text.str.startswith("+", na=False)
sprint_results_clean["race_total_seconds_from_text"] = pd.to_numeric(
    sprint_time_text.where(~sprint_is_gap).map(time_to_seconds), errors="raise")
sprint_results_clean["gap_to_winner_seconds"] = pd.to_numeric(
    sprint_time_text.where(sprint_is_gap).str.removeprefix("+").map(time_to_seconds), errors="raise")
display(sprint_results_clean[["time", "race_total_seconds_from_text", "gap_to_winner_seconds"]].head(10))


In [ ]:
sprint_results_clean["fastestLapTime_sec"] = pd.to_numeric(
    sprint_results_clean["fastestLapTime"].map(time_to_seconds), errors="raise")
display(sprint_results_clean[["fastestLapTime", "fastestLapTime_sec"]].head(10))


In [ ]:
# Verify each populated source value parsed; do not repeat conversions.
parsed_sprint_time = sprint_results_clean["race_total_seconds_from_text"].where(
    ~sprint_is_gap, sprint_results_clean["gap_to_winner_seconds"])
assert not (sprint_results_clean["time"].notna() & parsed_sprint_time.isna()).any()
assert not (sprint_results_clean["fastestLapTime"].notna() & sprint_results_clean["fastestLapTime_sec"].isna()).any()
print("Sprint time and fastestLapTime converted without losing populated values.")


In [ ]:
sprint_results_clean.info()

In [ ]:
sprint_results_clean.duplicated(
    subset=["raceId", "driverId"]
).sum()

In [ ]:
duplicates = sprint_results_clean[
    sprint_results_clean.duplicated(
        subset=["raceId", "driverId"],
        keep=False
    )
]

duplicates

**status.csv**

In [ ]:
status.info()

In [ ]:
status_clean = pd.read_csv(DATA_DIR / "status.csv",na_values=[r"\N"]) 

In [ ]:
status.isnull().sum()

In [ ]:
status.duplicated(subset=["statusId"]).sum()

### Remaining audit work

The supplied Hamza section ends here without a full three-table uniqueness/reference audit. This merge preserves the supplied inspections; it does not claim Objective 1 is complete for all tables. Add missing key checks, references, and before/after evidence before submission.
